In this notebook, we will construct the dataset that we will be working with throughout this project. Our data is derived from Wikipedia/Wikidata.

In [1]:
# Import the necessary libraries

import requests
import pandas as pd
import time
from datetime import datetime

In [2]:
# Wikipedia/Wikidata extraction definitions

HEADERS = {"User-Agent": "french-parliament-attention-project/1.0"}
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"
WIKI_PAGEVIEWS = (
    "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article"
    "/fr.wikipedia/all-access/all-agents/{title}/daily/{start}/{end}"
)

In [3]:
# Function to make sure the data requests are safe

def safe_request(url, params=None):
    for i in range(5):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=30)
            if r.status_code == 200:
                return r
            if r.status_code == 429:
                time.sleep(2 ** i)
            else:
                time.sleep(2)
        except Exception as e:
            print(f"  Request error: {e}")
            time.sleep(2)
    return None

In [4]:
# Using SPARQL, first step is to get the members of the 17th assembly and their characteristics (age, gender, parliamentary group)

# Step 1: MPs data extraction

query = """
SELECT DISTINCT ?dep ?depLabel ?sexLabel ?birthDate ?groupLabel ?frWikiTitle WHERE {
  ?dep p:P39 ?mandate .
  ?mandate ps:P39 wd:Q3044918 ;
           pq:P2937 wd:Q126471296 .

  FILTER NOT EXISTS { ?mandate pq:P582 ?end }

  OPTIONAL { ?dep wdt:P21 ?sex . }
  OPTIONAL { ?dep wdt:P569 ?birthDate . }

  OPTIONAL {
    ?mandate pq:P4100 ?group .
  }

  OPTIONAL {
    ?frWiki schema:about ?dep ;
            schema:isPartOf <https://fr.wikipedia.org/> ;
            schema:name ?frWikiTitle .
  }

  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "fr,en" .
  }
}
"""

r = safe_request(WIKIDATA_SPARQL, params={"query": query, "format": "json"})
if not r:
    raise RuntimeError("Could not reach Wikidata SPARQL endpoint.")
results = r.json()["results"]["bindings"]

# Step 2: Parsing the data into a dataframe

rows = []
for item in results:
    qid        = item["dep"]["value"].split("/")[-1]
    name       = item.get("depLabel", {}).get("value", "")
    gender_raw = item.get("sexLabel", {}).get("value", "")
    birth_raw  = item.get("birthDate", {}).get("value", "")
    group      = item.get("groupLabel", {}).get("value", "")
    wiki_title = item.get("frWikiTitle", {}).get("value", "")
    
    age = None # deriving age
    if birth_raw:
        try:
            bd  = datetime.strptime(birth_raw[:10], "%Y-%m-%d")
            ref = datetime(2025, 12, 31)
            age = ref.year - bd.year - ((ref.month, ref.day) < (bd.month, bd.day))
        except Exception:
            pass
            
    gl = gender_raw.lower() # normalizing gender
    if "féminin" in gl or "female" in gl:
        gender = "F"
    elif "masculin" in gl or "male" in gl:
        gender = "M"
    else:
        gender = None
        
    rows.append({
        "qid":        qid,
        "name":       name,
        "gender":     gender,
        "age":        age,
        "group":      group if group else None,
        "wiki_title": wiki_title if wiki_title else None,
    })
    
df_metadata = pd.DataFrame(rows)
df_metadata = df_metadata.sort_values("group", na_position="last")
df_metadata = df_metadata.drop_duplicates(subset="qid")
print(f" On a enfin {len(df_metadata)} MPs.")
df_metadata.head()

 On a enfin 574 MPs.


,qid,name,gender,age,group,wiki_title
79,Q136772110,Benoît Blanchard,M,64,Groupe Horizons et apparentés,Benoît Blanchard
175,Q30377001,Lise Magnier,F,41,Groupe Horizons et apparentés,Lise Magnier
75,Q130375116,Jean Moulliere,M,31,Groupe Horizons et apparentés,Jean Moulliere
122,Q127271701,Thomas Lam,M,44,Groupe Horizons et apparentés,Thomas Lam (homme politique)
25,Q112651528,Félicie Gérard,F,51,Groupe Horizons et apparentés,Félicie Gérard


In [5]:
# Before moving on to the next step, we must verify the cleanliness of the data already extracted

print("Structure des valeurs nulles:\n", df_metadata.isna().sum())
print("\nAge pas logique (<21 or >85):", df_metadata[(df_metadata["age"] < 21) | (df_metadata["age"] > 85)].shape[0])
print("\nGenre pas logique (ni M ni F):", df_metadata[~df_metadata["gender"].isin(["M", "F"])].shape[0])
print("\nDuplications QID:", df_metadata["qid"].duplicated().sum())

Structure des valeurs nulles:
 qid           0
name          0
gender        0
age           0
group         7
wiki_title    0
dtype: int64

Age pas logique (<21 or >85): 0

Genre pas logique (ni M ni F): 0

Duplications QID: 0


In [6]:
# The only issue is that there are seven members with no actual group, we must fix that

df_metadata[df_metadata["group"].isna() | (df_metadata["group"].str.strip() == "")][["name", "qid", "wiki_title"]]

,name,qid,wiki_title
59,Sabine Gervais,Q138728431,Sabine Gervais
110,Bernard Chaumeil,Q139031576,Bernard Chaumeil
185,Patricia Lemoine,Q57394738,Patricia Lemoine
372,Christopher Weissberg,Q112893812,Christopher Weissberg
426,Alexandra Martin,Q120798609,Alexandra Martin (1976)
516,Pauline Cestrieres,Q98235200,Pauline Cestrières
527,Patricia Maussion,Q63455916,Patricia Maussion


In [7]:
# Manual fix for the seven MPs

df_metadata.loc[df_metadata["qid"] == "Q98235200", "group"] = "groupe Renaissance"
df_metadata.loc[df_metadata["qid"] == "Q112893812", "group"] = "groupe Renaissance"
df_metadata.loc[df_metadata["qid"] == "Q120798609", "group"] = "groupe Les Républicains (Assemblée nationale)"
df_metadata.loc[df_metadata["qid"] == "Q63455916", "group"] = "groupe Renaissance"
df_metadata.loc[df_metadata["qid"] == "Q57394738", "group"] = "groupe Libertés, indépendance, outre-mer et territoires"
df_metadata.loc[df_metadata["qid"] == "Q138728431", "group"] = "non-inscrit à l'Assemblée nationale"
df_metadata.loc[df_metadata["qid"] == "Q139031576", "group"] = "non-inscrit à l'Assemblée nationale"

print("Structure des valeurs nulles:\n", df_metadata.isna().sum())
df_metadata["group"].value_counts()

Structure des valeurs nulles:
 qid           0
name          0
gender        0
age           0
group         0
wiki_title    0
dtype: int64


group
groupe Rassemblement national                              120
groupe Renaissance                                          95
groupe La France insoumise                                  71
groupe socialiste à l'Assemblée nationale                   65
groupe Les Républicains (Assemblée nationale)               47
groupe écologiste                                           38
groupe démocrate (MoDem et indépendants)                    34
Groupe Horizons et apparentés                               33
groupe Libertés, indépendance, outre-mer et territoires     25
groupe Gauche démocrate et républicaine                     17
Groupe Union des droites pour la République                 16
non-inscrit à l'Assemblée nationale                         13
Name: count, dtype: int64

In [8]:
# Now, we will get the pageviews for these MPs for the year 2025

def get_pageviews(title, year=2025):
    
    if not title:
        return None
    title = title.replace(" ", "_")
    url = WIKI_PAGEVIEWS.format(
        title=title,
        start=f"{year}010100",
        end=f"{year}123100",
    )

    r = safe_request(url)
    if not r:
        return None
    items = r.json().get("items", [])
    df_mp = pd.DataFrame(items)
    if df_mp.empty:
        return None
    df_mp["date"] = pd.to_datetime(df_mp["timestamp"].str[:8], format="%Y%m%d")
    df_mp = df_mp[["date", "views"]]
    df_mp["mp"] = title

    return df_mp

views = []
for _, row in df_metadata.iterrows():
    tmp = get_pageviews(row["wiki_title"])
    
    if tmp is not None:
        tmp["qid"] = row["qid"]    
        tmp["name"] = row["name"]   
        views.append(tmp)
df_views = pd.concat(views, ignore_index=True)

In [10]:
# Now, we must validate the cleanliness and fix any issues if any

print("Les MPs uniques:", df_views["qid"].nunique())
print("\nPlage des dates:", df_views["date"].min(), "→", df_views["date"].max())
print("\nStructure des valeurs nulles:\n", df_views.isna().sum())
print("\nDuplications:", df_views.duplicated(subset=["qid", "date"]).sum())
print("\nDescription statistique basique:\n", df_views["views"].describe())
print("\nDescription des dates par MP:\n", df_views.groupby("qid")["date"].nunique())
print("\nNombre de MPs par nombre de dates uniques::\n", df_views.groupby("qid")["date"].nunique().value_counts())

Les MPs uniques: 549

Plage des dates: 2025-01-01 00:00:00 → 2025-12-31 00:00:00

Structure des valeurs nulles:
 date     0
views    0
mp       0
qid      0
name     0
dtype: int64

Duplications: 0

Description statistique basique:
 count    196306.000000
mean        112.898811
std         605.529746
min           1.000000
25%          13.000000
50%          23.000000
75%          49.000000
max       73406.000000
Name: views, dtype: float64

Description des dates par MP:
 qid
Q104032664    365
Q106020252    365
Q106489533    365
Q106635078    365
Q107449933    365
             ... 
Q984375       365
Q98556479     365
Q99627308     365
Q99627615     365
Q99670641     365
Name: date, Length: 549, dtype: int64

Nombre de MPs par nombre de dates uniques::
 date
365    513
364     13
49       8
81       2
362      2
358      1
326      1
215      1
355      1
342      1
341      1
292      1
63       1
346      1
363      1
50       1
Name: count, dtype: int64


In [13]:
# As we saw, some MPs don't have 365 day coverage, only 513 members have full coverage. Here, we're studying if taking these 513 is representative of the 574 and not skewed or inclined

complete_qids = df_views.groupby("qid")["date"].nunique()
complete_qids = complete_qids[complete_qids == 365].index

df_513 = df_metadata[df_metadata["qid"].isin(complete_qids)]
df_574 = df_metadata.copy()

# Let's do a Chi-squared test to evaluate for gender and group (age is continuous, we can't)

from scipy.stats import chi2_contingency

gender_table = pd.crosstab(df_574["gender"], df_574["qid"].isin(complete_qids))
print("\nChi-Square - Gender p-value:", chi2_contingency(gender_table)[1])

group_table = pd.crosstab(df_574["group"], df_574["qid"].isin(complete_qids))
print("\nChi-Square - Group p-value:", chi2_contingency(group_table)[1])

# Statistical differences

print("\n\n- Analyse de genres:")
gender_tableau = pd.DataFrame({
    "574": df_574["gender"].value_counts(normalize=True),
    "513": df_513["gender"].value_counts(normalize=True)
})
print(gender_tableau)

print("\n\n- Analyse d'ages:")
age_tableau = pd.DataFrame({
    "574": df_574["age"].describe(),
    "513": df_513["age"].describe()
})
print(age_tableau)

print("\n\n- Analyse de groupes:")
group_tableau = pd.DataFrame({
    "574": df_574["group"].value_counts(normalize=True),
    "513": df_513["group"].value_counts(normalize=True)
})
print(group_tableau)


Chi-Square - Gender p-value: 0.08783610812121

Chi-Square - Group p-value: 0.08643546607775991


- Analyse de genres:
             574       513
gender                    
M       0.632404  0.645224
F       0.367596  0.354776


- Analyse d'ages:
              574         513
count  574.000000  513.000000
mean    50.794425   50.658869
std     11.979749   12.135249
min     25.000000   25.000000
25%     40.000000   40.000000
50%     52.000000   52.000000
75%     60.000000   60.000000
max     82.000000   82.000000


- Analyse de groupes:
                                                         574       513
group                                                                 
Groupe Horizons et apparentés                       0.057491  0.052632
Groupe Union des droites pour la République         0.027875  0.029240
groupe Gauche démocrate et républicaine             0.029617  0.027290
groupe La France insoumise                          0.123693  0.132554
groupe Les Républicains (Assemblé

The previous analysis tells us that yes the two sets are, relatively, highly similiar and the 513 can be used and isn't biased towards anything. So, let's save it as a CSV.

In [19]:
# Now, we must create the final dataset by joining the two dataframes for the 513 MPs

df_views = df_views.merge(
    df_metadata[["qid", "age", "gender", "group"]],
    on="qid",
    how="left"
)

df_views.head()

,date,views,mp,qid,name,age_x,gender_x,group_x,age_y,gender_y,group_y
0,2025-11-13,204,Benoît_Blanchard,Q136772110,Benoît Blanchard,64,M,Groupe Horizons et apparentés,64,M,Groupe Horizons et apparentés
1,2025-11-14,66,Benoît_Blanchard,Q136772110,Benoît Blanchard,64,M,Groupe Horizons et apparentés,64,M,Groupe Horizons et apparentés
2,2025-11-15,27,Benoît_Blanchard,Q136772110,Benoît Blanchard,64,M,Groupe Horizons et apparentés,64,M,Groupe Horizons et apparentés
3,2025-11-16,15,Benoît_Blanchard,Q136772110,Benoît Blanchard,64,M,Groupe Horizons et apparentés,64,M,Groupe Horizons et apparentés
4,2025-11-17,49,Benoît_Blanchard,Q136772110,Benoît Blanchard,64,M,Groupe Horizons et apparentés,64,M,Groupe Horizons et apparentés


In [22]:
# Create the final dataset and check validity

df_final = df_views[df_views["qid"].isin(complete_qids)].copy()

df_final["age"] = df_final[["age_x", "age_y"]].bfill(axis=1).iloc[:, 0]
df_final["gender"] = df_final[["gender_x", "gender_y"]].bfill(axis=1).iloc[:, 0]
df_final["group"] = df_final[["group_x", "group_y"]].bfill(axis=1).iloc[:, 0]

df_final = df_final.drop(columns=[
    "age_x", "age_y",
    "gender_x", "gender_y",
    "group_x", "group_y"
], errors="ignore")

df_final = df_final[[
    "qid",
    "mp",
    "name",
    "gender",
    "age",
    "group",
    "date",
    "views"
]]

df_final.head()

,qid,mp,name,gender,age,group,date,views
49,Q30377001,Lise_Magnier,Lise Magnier,F,41,Groupe Horizons et apparentés,2025-01-01,8
50,Q30377001,Lise_Magnier,Lise Magnier,F,41,Groupe Horizons et apparentés,2025-01-02,13
51,Q30377001,Lise_Magnier,Lise Magnier,F,41,Groupe Horizons et apparentés,2025-01-03,17
52,Q30377001,Lise_Magnier,Lise Magnier,F,41,Groupe Horizons et apparentés,2025-01-04,18
53,Q30377001,Lise_Magnier,Lise Magnier,F,41,Groupe Horizons et apparentés,2025-01-05,10


In [23]:
# Analyze the final dataset if all is good, so we can save as a CSV

print("Taille du dataset:", df_final.shape)

print("\nValeurs manquantes:\n", df_final.isna().sum())

print("\nPlage des dates:", df_final["date"].min(), "→", df_final["date"].max())

print("\nNombre de MPs uniques:", df_final["qid"].nunique())

print("\nNombre de jours par MP:")
print(df_final.groupby("qid")["date"].nunique().value_counts().sort_index())

print("\nDuplications (qid, date):", df_final.duplicated(subset=["qid", "date"]).sum())

print("\nAnalyse du genre par MP:")
print(df_final.groupby("qid")["gender"].nunique().value_counts())

print("\nAnalyse de l'age par MP:")
print(df_final.groupby("qid")["age"].nunique().value_counts())

print("\nAnalyse du groupe par MP:")
print(df_final.groupby("qid")["group"].nunique().value_counts())

print("\nAnalyse statistique des views:\n", df_final["views"].describe())

Taille du dataset: (187245, 8)

Valeurs manquantes:
 qid       0
mp        0
name      0
gender    0
age       0
group     0
date      0
views     0
dtype: int64

Plage des dates: 2025-01-01 00:00:00 → 2025-12-31 00:00:00

Nombre de MPs uniques: 513

Nombre de jours par MP:
date
365    513
Name: count, dtype: int64

Duplications (qid, date): 0

Analyse du genre par MP:
gender
1    513
Name: count, dtype: int64

Analyse de l'age par MP:
age
1    513
Name: count, dtype: int64

Analyse du groupe par MP:
group
1    513
Name: count, dtype: int64

Analyse statistique des views:
 count    187245.000000
mean        117.412865
std         619.380850
min           1.000000
25%          14.000000
50%          23.000000
75%          52.000000
max       73406.000000
Name: views, dtype: float64


In [24]:
# Final step of data construction, save to CSV

df_final.to_csv("dataset.csv", index=False)